# Laboratorio — Robot de entregas en un almacén

A partir de la **imagen**, construye el MDP y resuélvelo con **Value Iteration** y **Policy Iteration**.

![Mundo del ejercicio](https://drive.google.com/uc?export=view&id=1_sJaD57gHuiz1joEgl4B-u0aDy8jtMDo)



## Convención y notación

$$
s=(row,col)
$$

$$
T(s,a,s')=P(s'\mid s,a)
$$

$$
R(s)
$$

Para Value Iteration:

$$
V_{k+1}(s)
=
R(s)
+
\gamma
\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$

Para Policy Evaluation:

$$
V_{k+1}^{\pi}(s)
=
R(s)
+
\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Acciones

```python
UP    = (-1, 0)
DOWN  = ( 1, 0)
LEFT  = ( 0,-1)
RIGHT = ( 0, 1)
```



## Reglas del mundo

El grid tiene **5 filas × 6 columnas**.

### Estados especiales

A partir de la imagen identifica:

- `START`
- estanterías / paredes;
- zona de entrega `+10` (**terminal**);
- estación de carga `+2` (**terminal**);
- peligro mortal `-10` (**terminal**);
- peligros `-3` (**no terminales**);
- celdas de piso resbaloso.

### Recompensa

Usamos la convención del notebook de clase, es decir, **\(R(s)\)**:

- entrega: `+10`;
- carga: `+2`;
- peligro mortal: `-10`;
- peligro: `-3`;
- cualquier otro estado transitable: `-1` (costo por paso).

### Dinámica

La transición depende del **estado actual**:

**Piso normal**

$$
P(\text{dirección elegida})=0.90
$$

$$
P(\text{desviación izquierda})=0.05
$$

$$
P(\text{desviación derecha})=0.05
$$

**Piso resbaloso**

$$
P(\text{dirección elegida})=0.60
$$

$$
P(\text{desviación izquierda})=0.20
$$

$$
P(\text{desviación derecha})=0.20
$$

Si el movimiento sale del grid o golpea una estantería, el robot **permanece en el mismo estado**.

Usa:

$$
\gamma=0.9,\qquad \theta=10^{-4}
$$



## Parte 1 — Modela el MDP

Completa la clase `WarehouseMDP`.

La parte importante no es escribir muchas líneas de código: es traducir correctamente la imagen a:

- estados;
- acciones;
- recompensas;
- terminales;
- obstáculos;
- tipos de piso;
- función de transición.


In [1]:
import sys
!{sys.executable} -m pip install numpy
import numpy as np

class WarehouseMDP:
    def __init__(self):
        self.height = 5
        self.width = 6

        # TODO: completa a partir de la imagen
        self.start = (0,0)
        self.walls = {(0,3),(1,1),(2,4),(4,2)}
        self.slippery_states = {(1,2),(2,1),(3,3)}

        self.terminal_states = {
            (0, 5): 10,
            (3, 5): -10,
            (2, 2): 2,
        }

        self.danger_states = {
            (1, 4): -3,
            (4, 1): -3,
        }

        self.living_reward = -1.0
        self.gamma = 0.9

        self.actions = [
            (-1, 0),  # UP
            ( 1, 0),  # DOWN
            ( 0,-1),  # LEFT
            ( 0, 1),  # RIGHT
        ]

    def is_valid_state(self, state):
        # TODO
        r,c =state
        if r < 0 or r >= self.height or c < 0 or c >= self.width:
            return False
        if state in self.walls:
            return False
        return True
        

    def states(self):
        # TODO
        return [
        (r, c)
        for r in range(self.height)
        for c in range(self.width)
        if (r, c) not in self.walls
        ]

    def is_terminal(self, state):
        # TODO
        return state in self.terminal_states
        

    def get_reward(self, state):
        # TODO: implementa R(s)
        if state in self.terminal_states:
            return self.terminal_states[state]
        if state in self.danger_states:
            return self.danger_states[state]
        return self.living_reward

    def get_transition_probs(self, state, action):
        """
        Devuelve:
            [(next_state, probability), ...]

        Recuerda:
        - las probabilidades dependen de si 'state' es resbaloso;
        - si golpea pared/borde, next_state = state.
        """
        # TODO
        if self.is_terminal(state):
            return [(state, 1.0)]

        if state in self.slippery_states:
            p_intended, p_left, p_right = 0.60, 0.20, 0.20
        else:
            p_intended, p_left, p_right = 0.90, 0.05, 0.05

        if action == (-1, 0):        # UP
            left_action, right_action = (0, -1), (0, 1)
        elif action == (1, 0):       # DOWN
            left_action, right_action = (0, 1), (0, -1)
        elif action == (0, -1):      # LEFT
            left_action, right_action = (1, 0), (-1, 0)
        elif action == (0, 1):       # RIGHT
            left_action, right_action = (-1, 0), (1, 0)

        r, c = state

        # Dirección elegida
        dr, dc = action
        next_intended = (r + dr, c + dc)
        if not self.is_valid_state(next_intended):
            next_intended = state

        # Desviación izquierda
        dr, dc = left_action
        next_left = (r + dr, c + dc)
        if not self.is_valid_state(next_left):
            next_left = state

        # Desviación derecha
        dr, dc = right_action
        next_right = (r + dr, c + dc)
        if not self.is_valid_state(next_right):
            next_right = state

        outcomes = [
            (next_intended, p_intended),
            (next_left, p_left),
            (next_right, p_right),
        ]

        merged = {}
        for ns, p in outcomes:
            merged[ns] = merged.get(ns, 0.0) + p

        return list(merged.items())



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: pip install --upgrade pip



### Validación mínima del modelo

Antes de implementar Bellman, valida primero el MDP.


In [2]:
grid = WarehouseMDP()

S = grid.states()
print("Número de estados:", len(S))

# Cada distribución T(s,a,·) debe sumar 1.
for s in S:
    for a in grid.actions:
        transitions = grid.get_transition_probs(s, a)
        total = sum(p for _, p in transitions)
        assert abs(total - 1.0) < 1e-12

print("✓ Todas las distribuciones de transición suman 1.")


Número de estados: 26
✓ Todas las distribuciones de transición suman 1.



## Parte 2 — Value Iteration

Implementa:

$$
V_{k+1}(s)
=
R(s)+\gamma\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$


In [8]:
def expected_next_value(grid, state, action, V):
    # TODO:
    transitions = grid.get_transition_probs(state, action)
    total = 0.0
    for next_state, prob in transitions:
        contrib = prob * V[next_state]
        print(f"      + P={prob:.2f} * V{next_state}={V[next_state]:.4f}  -> {contrib:.4f}")
        total += prob * V[next_state]
    return total
    


def value_iteration(grid, threshold=1e-4, max_iter=10_000):
    # TODO
    V = {s: 0.0 for s in grid.states()}

    for iteration in range(1, max_iter + 1):
        V_new = {}
        delta = 0.0

        for s in grid.states():
            if grid.is_terminal(s):
                V_new[s] = grid.get_reward(s)
            else:
                V_new[s] = max(
                    grid.get_reward(s) + grid.gamma * expected_next_value(grid, s, a, V)
                    for a in grid.actions
                )

            delta = max(delta, abs(V_new[s] - V[s]))

        V = V_new

        if delta < threshold:
            break

    return V, iteration


def extract_policy(grid, V):
    # TODO:
    policy = {}

    for s in grid.states():
        if grid.is_terminal(s):
            policy[s] = None
            continue

        policy[s] = max(
            grid.actions,
            key=lambda a: grid.get_reward(s) + grid.gamma * expected_next_value(grid, s, a, V)
        )

    return policy



## Parte 3 — Policy Iteration

### Policy Evaluation

$$
V_{k+1}^{\pi}(s)
=
R(s)+\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Policy Improvement

$$
\pi_{\mathrm{new}}(s)
=
\arg\max_a
\sum_{s'}T(s,a,s')V^\pi(s')
$$

In [ ]:
def policy_evaluation(grid, policy, threshold=1e-4, max_iter=10_000):
    # TODO
    V = {s: 0.0 for s in grid.states()}

    for iteration in range(1, max_iter + 1):
        V_new = {}
        delta = 0.0

        for s in grid.states():
            if grid.is_terminal(s):
                V_new[s] = grid.get_reward(s)
            else:
                a = policy[s] 
                V_new[s] = grid.get_reward(s) + grid.gamma * expected_next_value(grid, s, a, V)

            delta = max(delta, abs(V_new[s] - V[s]))

        V = V_new

        if delta < threshold:
            break

    return V
    


def policy_improvement(grid, V):
    # TODO
    policy = {}

    for s in grid.states():
        if grid.is_terminal(s):
            policy[s] = None
            continue

        policy[s] = max(
            grid.actions,
            key=lambda a: expected_next_value(grid, s, a, V)
        )

    return policy
    pass


def policy_iteration(grid, threshold=1e-4, max_iter=100):
    # TODO:
    policy = {s: grid.actions[0] for s in grid.states()}
    history = []

    for i in range(max_iter):
        # evaluacion
        V = policy_evaluation(grid, policy, threshold=threshold)

        # mejora
        new_policy = policy_improvement(grid, V)

        history.append(i + 1)

        if new_policy == policy:
            break

        policy = new_policy

    return policy, V, history
    pass



## Parte 4 — Visualización y comparación


In [16]:
ARROWS = {
    (-1, 0): "↑",
    ( 1, 0): "↓",
    ( 0,-1): "←",
    ( 0, 1): "→",
}

def print_values(grid, V):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)
            if s in grid.walls:
                row.append("  WALL  ")
            else:
                row.append(f"{V[s]:+7.3f}")
        print(" | ".join(row))


def print_policy(grid, policy):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)

            if s in grid.walls:
                row.append(" # ")
            elif grid.is_terminal(s):
                reward = grid.get_reward(s)
                row.append(f"{reward:+.0f}")
            else:
                row.append(f" {ARROWS[policy[s]]} ")

        print(" | ".join(row))


In [17]:
# VALUE ITERATION
V_vi, n_vi = value_iteration(grid)
pi_vi = extract_policy(grid, V_vi)

print("=== VALUE ITERATION ===")
print("Iteraciones:", n_vi)
print("\nValores:")
print_values(grid, V_vi)
print("\nPolítica:")
print_policy(grid, pi_vi)


# POLICY ITERATION
pi_pi, V_pi, history = policy_iteration(grid)

print("\n=== POLICY ITERATION ===")
print("Historia:", history)
print("\nValores:")
print_values(grid, V_pi)
print("\nPolítica:")
print_policy(grid, pi_pi)

assert pi_vi == pi_pi
print("\n✓ Ambos algoritmos encontraron la misma política óptima.")


      + P=0.95 * V(0, 0)=0.0000  -> 0.0000
      + P=0.05 * V(0, 1)=0.0000  -> 0.0000
      + P=0.90 * V(1, 0)=0.0000  -> 0.0000
      + P=0.05 * V(0, 1)=0.0000  -> 0.0000
      + P=0.05 * V(0, 0)=0.0000  -> 0.0000
      + P=0.95 * V(0, 0)=0.0000  -> 0.0000
      + P=0.05 * V(1, 0)=0.0000  -> 0.0000
      + P=0.90 * V(0, 1)=0.0000  -> 0.0000
      + P=0.05 * V(0, 0)=0.0000  -> 0.0000
      + P=0.05 * V(1, 0)=0.0000  -> 0.0000
      + P=0.90 * V(0, 1)=0.0000  -> 0.0000
      + P=0.05 * V(0, 0)=0.0000  -> 0.0000
      + P=0.05 * V(0, 2)=0.0000  -> 0.0000
      + P=0.90 * V(0, 1)=0.0000  -> 0.0000
      + P=0.05 * V(0, 2)=0.0000  -> 0.0000
      + P=0.05 * V(0, 0)=0.0000  -> 0.0000
      + P=0.90 * V(0, 0)=0.0000  -> 0.0000
      + P=0.10 * V(0, 1)=0.0000  -> 0.0000
      + P=0.90 * V(0, 2)=0.0000  -> 0.0000
      + P=0.10 * V(0, 1)=0.0000  -> 0.0000
      + P=0.95 * V(0, 2)=0.0000  -> 0.0000
      + P=0.05 * V(0, 1)=0.0000  -> 0.0000
      + P=0.90 * V(1, 2)=0.0000  -> 0.0000
      + P=0


## Parte 5 — Interpreta la política

Antes de cambiar parámetros, responde:

1. Desde `START`, ¿el robot busca la **entrega +10** o prefiere la **estación de carga +2**?
2. ¿Por qué una recompensa menor podría ser óptima?
3. ¿En qué estados el piso resbaloso cambia la decisión?
4. ¿Qué papel cumple el costo por paso `-1`?
5. ¿Por qué \(T(s,a,s')\) ya no puede implementarse con las mismas probabilidades para todos los estados?

### Experimento A — Menos costo por paso

Cambia:

```python
living_reward = -0.1
```

Predice la política **antes de ejecutar**.

### Experimento B — Piso muy resbaloso

Cambia la probabilidad de movimiento deseado del piso resbaloso:

```python
0.60 → 0.40
```

y reparte el restante entre las dos desviaciones.

### Experimento C — Más paciencia

Cambia:

```python
gamma = 0.99
```

¿La política valora más la recompensa `+10` distante?

### Bonus

Encuentra aproximadamente el valor de `living_reward` a partir del cual la política desde `START` cambia entre:

- ir a carga `+2`;
- intentar llegar a entrega `+10`.
